# Fine-tuning no Google Colab

Notebook-base para executar o treino a partir de `resources/finetuning_qa.jsonl` enviado manualmente ao Google Drive.

Fluxo:
1. Instalar dependências
2. Montar o Google Drive
3. Carregar o dataset JSONL
4. Configurar o modelo base
5. Executar o fine-tuning leve com QLoRA
6. Fazer merge do adapter no modelo base e salvar o modelo completo
7. Avaliar o modelo fine-tuned no conjunto de teste


In [ ]:
!pip -q install --upgrade "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip -q install --upgrade transformers datasets accelerate peft trl bitsandbytes sentencepiece huggingface_hub evaluate bert-score


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
assert HF_TOKEN, 'Defina o Secret HF_TOKEN no Colab'
login(token=HF_TOKEN)

DATASET_PATH = Path('/content/drive/MyDrive/teach-chalenge3/finetuning_qa.jsonl')
OUTPUT_DIR = Path('/content/drive/MyDrive/teach-chalenge3/medqa-finetuned-model')
BASE_MODEL = 'meta-llama/Llama-3.2-1B-Instruct'
MAX_LENGTH = 512

assert DATASET_PATH.exists(), f'Dataset não encontrado em {DATASET_PATH}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(DATASET_PATH)
print(OUTPUT_DIR)


In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported

max_seq_length = MAX_LENGTH
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print(model.__class__.__name__)
print(tokenizer.__class__.__name__)


In [ ]:
from datasets import load_dataset

dataset = load_dataset('json', data_files=str(DATASET_PATH), split='train')
dataset = dataset.train_test_split(test_size=0.05, seed=42)
dataset


In [ ]:
def preprocess_function(examples):
    tokenized = tokenizer(
        examples['text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )
    tokenized['labels'] = tokenized['input_ids'].copy()
    tokenized['length'] = [len(ids) for ids in tokenized['input_ids']]
    return tokenized

tokenized_dataset = dataset.map(preprocess_function, batched=True, remove_columns=['source'])
tokenized_dataset = tokenized_dataset.filter(lambda example: example['length'] > 0)
tokenized_dataset


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
    max_seq_length=max_seq_length,
)


In [ ]:
from pathlib import Path
from trl import SFTTrainer
from transformers import TrainingArguments
from transformers.trainer_utils import get_last_checkpoint

SAVE_STEPS = 100

use_bf16 = is_bfloat16_supported()
last_checkpoint = get_last_checkpoint(str(OUTPUT_DIR)) if OUTPUT_DIR.exists() else None

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=25,
    save_strategy='steps',
    save_steps=SAVE_STEPS,
    save_total_limit=1,
    eval_strategy='epoch',
    bf16=use_bf16,
    fp16=not use_bf16,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['test'],
    processing_class=tokenizer,
)


In [ ]:
trainer.train(resume_from_checkpoint=last_checkpoint)
model.save_pretrained_merged(str(OUTPUT_DIR), tokenizer, save_method='merged_16bit')


## Avaliação no conjunto de teste

A avaliação final usa somente uma amostra embaralhada de `dataset['test']`, que não foi usada para atualizar os pesos. O modelo fine-tuned gera respostas em batches, e a biblioteca Hugging Face `evaluate` calcula Exact Match, Token F1 e ROUGE-L; `BERTScore` mede similaridade semântica.

In [ ]:
import torch
from evaluate import load
from tqdm.auto import tqdm
from transformers import GenerationConfig

model.generation_config = GenerationConfig.from_pretrained(str(OUTPUT_DIR), local_files_only=True)
model.generation_config.max_new_tokens = 128
model.generation_config.do_sample = False
model.generation_config.repetition_penalty = 1.05
model.generation_config.temperature = 1.0
model.generation_config.top_p = None

# Uma amostra torna a avaliação de geração viável no Colab; aumente se necessário.
EVAL_SIZE = 500
EVAL_BATCH_SIZE = 4
eval_dataset = dataset['test'].shuffle(seed=3407).select(range(min(EVAL_SIZE, len(dataset['test']))))

def split_test_record(text):
    prompt, reference = text.split('[|Answer|]', 1)
    reference = reference.split('[|eAnswer|]', 1)[0].strip()
    return f'{prompt}[|Answer|]', reference

prompts, references = zip(*(split_test_record(record['text']) for record in eval_dataset))
predictions = []
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.eval()
for start in tqdm(range(0, len(prompts), EVAL_BATCH_SIZE), desc='Avaliando'):
    batch_prompts = prompts[start:start + EVAL_BATCH_SIZE]
    inputs = tokenizer(list(batch_prompts), return_tensors='pt', padding=True,
                       truncation=True, max_length=MAX_LENGTH).to(model.device)
    with torch.inference_mode():
        outputs = model.generate(**inputs)
    generated_ids = outputs[:, inputs['input_ids'].shape[1]:]
    batch_predictions = tokenizer.batch_decode(
        generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    predictions.extend(prediction.split('[|eAnswer|]', 1)[0].strip()
                      for prediction in batch_predictions)

references = list(references)
assert references, 'O conjunto de teste está vazio'

rouge = load('rouge')
squad = load('squad')
bertscore = load('bertscore')
ids = [str(index) for index in range(len(references))]
qa_predictions = [{'id': item_id, 'prediction_text': prediction}
                 for item_id, prediction in zip(ids, predictions)]
qa_references = [{'id': item_id, 'answers': {'text': [reference], 'answer_start': [0]}}
                 for item_id, reference in zip(ids, references)]

rouge_scores = rouge.compute(predictions=predictions, references=references, use_stemmer=True)
qa_scores = squad.compute(predictions=qa_predictions, references=qa_references)
bert_scores = bertscore.compute(predictions=predictions, references=references,
                               lang='en', model_type='distilbert-base-uncased')

print(f'Test examples evaluated: {len(references)}')
print(f'Exact match: {qa_scores["exact_match"] / 100:.4f}')
print(f'Token F1: {qa_scores["f1"] / 100:.4f}')
print(f'ROUGE-L: {rouge_scores["rougeL"]:.4f}')
print(f'BERTScore F1: {sum(bert_scores["f1"]) / len(bert_scores["f1"]):.4f}')
for index in range(min(3, len(predictions))):
    print(f'\nExample {index + 1}')
    print(f'Reference: {references[index]}')
    print(f'Prediction: {predictions[index]}')
